# Video 01 - What is an AI agent? + your first LLM call

**You are here:** Video 1 of 13 (see `../you-are-here.svg`)

**The one idea:** an LLM is a function - **text in, text out**.

**By the end you can:**
1. Explain the difference between an LLM, a chatbot and an agent.
2. Call an LLM from this notebook and print its reply.
3. Explain why an "OpenAI" client is talking to a Llama model in your Databricks workspace.

**Before you start:** attach this notebook to **Serverless** (top right) and check [setup.md](../../setup.md).

## Part 1 - Connect to the model
Three lines: get a client that is already signed in as you, and pick a model by name.

In [0]:
from databricks.sdk import WorkspaceClient

# A ready-to-use client, signed in as you. No API keys needed.
client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Which model to talk to (a "serving endpoint" in your workspace)
MODEL = "databricks-meta-llama-3-3-70b-instruct"

print("Connected. Using model:", MODEL)

In [0]:
# Hello World: one message in, one reply out
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
)

print(response.choices[0].message.content)

## What just happened?
* We sent **text in** (our question) and got **text out** (the reply). That's all an LLM does.
* The reply text lives at `response.choices[0].message.content`. Looks like a mouthful - Video 03 unpacks every piece. For now: that's where the text is.
* The `messages=[...]` list and the word `"role"` are explained in Video 02.

**Your turn:** change `question` below to anything you like and run it.

In [0]:
question = "Give me 3 everyday examples of things an AI agent could do for me."

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
print(response.choices[0].message.content)

## Part 2 - Where does the request actually go?
We used an `openai` client, but we are **not** using OpenAI. "OpenAI-compatible" is a message **format**. Databricks speaks it, so the client is pointed at **your workspace**, which sends the request to a model it hosts (diagram: `../diagrams/03-where-the-request-goes.svg`).

Run the next cell to see it for yourself.

In [0]:
print("Client type     :", type(client).__name__)   # a normal OpenAI client...
print("Sends requests to:", client.base_url)       # ...pointed at YOUR workspace
print("Model / endpoint :", MODEL)

In [0]:
# Same question, different model. Only the model name changes.
OTHER_MODEL = "databricks-claude-haiku-4-5"   # pick any chat model listed under Serving

for model in [MODEL, OTHER_MODEL]:
    reply = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
    )
    print(f"--- {model} ---")
    print(reply.choices[0].message.content, "\n")

## Cost meter
Models read and write **tokens** - chunks of text, roughly 3/4 of a word. You pay per token, and input (what you send) and output (what the model writes) are usually priced differently.

The prices below are **example numbers** so you can see the maths. Check your real rate on the Databricks Foundation Model pricing page and update them.

In [0]:
# EXAMPLE prices in US dollars per 1 million tokens - replace with your real rate
PRICE_PER_1M_INPUT = 0.50
PRICE_PER_1M_OUTPUT = 1.50

usage = response.usage
cost = usage.prompt_tokens / 1_000_000 * PRICE_PER_1M_INPUT + usage.completion_tokens / 1_000_000 * PRICE_PER_1M_OUTPUT

print(f"Tokens in : {usage.prompt_tokens}")
print(f"Tokens out: {usage.completion_tokens}")
print(f"Estimated cost of the last call: ${cost:.6f}")
print(f"Calls like that per $1: about {int(1 / cost):,}")

## What could go wrong? Confidently wrong answers
An LLM predicts text - it does not *calculate*. Ask it to multiply two big numbers and it will answer **confidently and wrongly**, often with a different wrong answer each time. Stating something false with confidence is called a **hallucination**.

**Predict first:** will the model get `48729 * 3817` right?

In [0]:
question = "What is 48729 * 3817? Answer with just the number."

for attempt in range(1, 4):
    answer = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
    ).choices[0].message.content
    print(f"Attempt {attempt} - model says: {answer}")

print(f"Python says          : {48729 * 3817}")
# Agents fix this with TOOLS: the model asks your code to run a calculator (Videos 06-08).

## Recap
* An **LLM** is a function: text in, text out.
* A **chatbot** talks. An **agent** talks **and acts** - it uses tools in a loop until the goal is reached.
* You called an LLM on Databricks. The `openai` client is just the format; the model runs in your workspace.
* LLMs can be **confidently wrong**. Tools will fix that.

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `01-hello-llm-solutions` (try first!).

**Next:** Video 02 - Talking to the model: messages and roles.